# Week 1: Dataset and Preprocessing
**Verstech Internship Program | Track 1: AI / Machine Learning**

**Project:** Customer Churn Prediction
**Dataset:** Telco Customer Churn (Kaggle: `blastchar/telco-customer-churn`)
**Goal:** Load the dataset, understand it, clean it, and prepare it for machine learning.

**Steps in this notebook**
1. Setup and imports
2. Load the dataset
3. Understand the data
4. Clean the data (missing values, duplicates, data types)
5. Encode text columns
6. Split, then scale
7. Save the cleaned dataset and scaler

## 1. Setup and imports

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)
print("Libraries imported")

Libraries imported


## 2. Load the dataset
Download the CSV from Kaggle (search **Telco Customer Churn**) and keep it in the same folder as this notebook.
The file name is usually `WA_Fn-UseC_-Telco-Customer-Churn.csv`.

In **Google Colab**, the cell below opens an upload box. In **Jupyter / VS Code** it just reads the file from the folder.

In [2]:
FILE_NAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"   # change if your file name is different

if not os.path.exists(FILE_NAME):
    try:
        from google.colab import files      # only works in Colab
        uploaded = files.upload()
        FILE_NAME = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"Put '{FILE_NAME}' in the same folder as this notebook.")

df = pd.read_csv(FILE_NAME)
print("Loaded:", FILE_NAME)

Loaded: WA_Fn-UseC_-Telco-Customer-Churn.csv


## 3. Understand the data

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
print('Rows and columns:', df.shape)

Rows and columns: (7043, 21)


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


### Column meanings
| Column | Meaning |
|---|---|
| customerID | Unique ID of the customer (not useful for prediction) |
| gender, SeniorCitizen, Partner, Dependents | Customer demographics |
| tenure | Number of months the customer has stayed |
| PhoneService, MultipleLines | Phone service details |
| InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies | Internet service and add-ons |
| Contract, PaperlessBilling, PaymentMethod | Contract and billing details |
| MonthlyCharges, TotalCharges | Amount charged per month and in total |
| **Churn** | **Target: did the customer leave? (Yes / No)** |

In [7]:
target = "Churn"
print("Target column:", target)
print(df[target].value_counts())
print()
print(df[target].value_counts(normalize=True).round(3))

Target column: Churn
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


The target is imbalanced (about 73% No, 27% Yes). Because of this, accuracy alone can mislead in Week 4, so we will use a **stratified** split below.

## 4. Clean the data

### 4.1 Fix data types
`TotalCharges` should be numeric, but it is read as text because a few rows contain blank spaces.

In [8]:
print("TotalCharges dtype before:", df["TotalCharges"].dtype)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")   # blanks become NaN
print("TotalCharges dtype after: ", df["TotalCharges"].dtype)

TotalCharges dtype before: str
TotalCharges dtype after:  float64


### 4.2 Missing values

In [9]:
missing = df.isnull().sum()
missing[missing > 0]

TotalCharges    11
dtype: int64

In [10]:
# These customers have tenure = 0 (new customers, not billed yet)
df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,NaN
753,3115-CZMZD,0,20.25,NaN
936,5709-LVOEQ,0,80.85,NaN
1082,4367-NUYAO,0,25.75,NaN
1340,1371-DWPAZ,0,56.05,NaN
3331,7644-OMVMY,0,19.85,NaN
3826,3213-VVOLG,0,25.35,NaN
4380,2520-SGTTA,0,20.00,NaN
5218,2923-ARZLG,0,19.70,NaN
6670,4075-WKNIU,0,73.35,NaN


In [11]:
# Numeric column: fill with the median
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Text columns: fill with the mode (none are missing here, but this keeps the code safe)
for col in df.select_dtypes(include="object").columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values left:", df.isnull().sum().sum())

Missing values left: 0


/tmp/ipykernel_95/1760336416.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


### 4.3 Drop columns that do not help prediction
`customerID` is just a unique label.

In [12]:
df = df.drop("customerID", axis=1)
df.shape

(7043, 20)

### 4.4 Duplicates

In [13]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

Duplicate rows: 22
Shape after removing duplicates: (7021, 20)


### 4.5 Clean category values
Some columns use `No internet service` or `No phone service`. These mean the same as `No`, so we merge them to avoid extra columns.

In [14]:
cols_to_fix = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
               "TechSupport", "StreamingTV", "StreamingMovies"]
for col in cols_to_fix:
    df[col] = df[col].replace({"No internet service": "No", "No phone service": "No"})

# SeniorCitizen is 0/1, make it Yes/No like the other columns (it is encoded later)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# Strip any extra spaces in text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df.head()

/tmp/ipykernel_95/3090004680.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,No,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,No,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 5. Encode text columns
Models only understand numbers.
- **Target** (`Churn`): Yes = 1, No = 0
- **Other text columns:** one-hot encoding with `pd.get_dummies(drop_first=True)`

In [15]:
df[target] = df[target].map({"Yes": 1, "No": 0})

cat_cols = df.select_dtypes(include="object").columns.tolist()
print("Columns to encode:", cat_cols)

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

Columns to encode: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


Shape after encoding: (7021, 24)


/tmp/ipykernel_95/1313944272.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns.tolist()


,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,SeniorCitizen_Yes,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_Yes,OnlineBackup_Yes,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,29.85,29.85,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0
1,34,56.95,1889.50,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,1
2,2,53.85,108.15,1,1,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0,0,1
3,45,42.30,1840.75,0,1,0,0,0,0,0,0,0,1,0,1,1,0,0,1,0,0,0,0,0
4,2,70.70,151.65,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0


In [16]:
# Final check before saving
print("Missing values:", df_encoded.isnull().sum().sum())
print("Non-numeric columns:", df_encoded.select_dtypes(exclude="number").columns.tolist())

Missing values: 0
Non-numeric columns: []


## 6. Save the cleaned dataset
Saved **before scaling**, so it can be used again in Week 2 (EDA) and Week 3 (modeling).

In [17]:
# Readable cleaned version (for EDA in Week 2)
df.to_csv("data/processed/telco_cleaned.csv", index=False)

# Encoded version (ready for models)
df_encoded.to_csv("data/processed/telco_encoded.csv", index=False)
print("Saved cleaned and encoded CSV files in data/processed/")

Saved cleaned and encoded CSV files in data/processed/


## 7. Split, then scale
**Split first, then scale.** The scaler is fitted on training data only. Otherwise the test data leaks into training (data leakage).

In [18]:
X = df_encoded.drop(target, axis=1)
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Churn rate in train:", round(y_train.mean(), 3), "| in test:", round(y_test.mean(), 3))

X_train: (5616, 23) | X_test: (1405, 23)
Churn rate in train: 0.264 | in test: 0.265


In [19]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit ONLY on training data
X_test_scaled = scaler.transform(X_test)         # only transform the test data

# Quick check: training data should now have mean ~0 and std ~1
print("Train mean (approx):", X_train_scaled.mean().round(3))
print("Train std  (approx):", X_train_scaled.std().round(3))

Train mean (approx): -0.0
Train std  (approx): 1.0


In [20]:
# Save the scaler and the split data for the next weeks
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(list(X.columns), "models/feature_columns.pkl")

X_train.assign(Churn=y_train.values).to_csv("data/processed/train.csv", index=False)
X_test.assign(Churn=y_test.values).to_csv("data/processed/test.csv", index=False)
print("Saved scaler, feature list, train.csv and test.csv")

Saved scaler, feature list, train.csv and test.csv


## Summary
- Loaded the Telco Customer Churn dataset (7,043 rows, 21 columns) and identified `Churn` as the target.
- Fixed `TotalCharges` (text to numeric) and filled the 11 missing values with the median.
- Dropped `customerID`, removed duplicates, and merged `No internet service` / `No phone service` into `No`.
- Encoded the target (Yes/No to 1/0) and the other text columns with one-hot encoding.
- Split the data 80/20 (stratified), then scaled using a scaler fitted on training data only.

**Deliverables:** cleaned dataset (`data/processed/telco_cleaned.csv`, `telco_encoded.csv`) and this preprocessing notebook.

**Next week:** Exploratory Data Analysis using `telco_cleaned.csv`.